# Step 1. Company pick at random
Question: Which mid-size WC company do I build test exhibits from at random? 

Criteria (known cells only, so the pick uses no hindsight):
- **Complete (all 10 AYs):** full triangle, so every maturity gets graded and every AY feeds the factor averages.
- **Size between p25 and p75 of median net EP per AY:** small lines' factors are 2–3.5× noisier (probe 8). The middle half is typical of the sample, so it's not dominated by noise or by one large book.
- **Ceded swing ≤ 0.5:** a changing reinsurance program can change net development (excess of loss speeds it up). That would add a chain ladder error that has nothing to do with the method (probe 7).
- **No negative case cells (case ≤ −2):** reported below paid distorts the reported triangle's factors (probe 3). A case of −1 is $000 rounding, so it's allowed.

Random pick among eligible lines, seed fixed before running: I can't steer it toward a company that flatters a method, and the seed makes the pick reproducible.

In [15]:
# Phase 2, step 1: random pick of one mid-size WC company (known cells only)
import numpy as np
import pandas as pd

SEED = 20261102  # fixed before running; run once, never re-roll

wc_clean = pd.read_csv("../data/clean/wc_clean.csv")

df = wc_clean.copy()  # 60 clean WC lines
known = df[df["AccidentYear"] + df["DevelopmentLag"] - 1 <= 2007].copy()
known["case"] = known["IncurredLosses"] - known["BulkLoss"] - known["CumPaidLoss"]
prem = known[known["DevelopmentLag"] == 1]  # one row per company-AY

ay = prem.groupby("GRCODE")
lines = pd.DataFrame({
    "GRNAME": ay["GRNAME"].first(),
    "n_ay": ay["AccidentYear"].nunique(),
    "size": ay["EarnedPremNet"].median(),  # $000, median net EP per AY
    "swing": (prem["EarnedPremCeded"] / prem["EarnedPremDIR"])
             .groupby(prem["GRCODE"]).agg(lambda s: s.max() - s.min()),
    "neg_case": (known["case"] <= -2).groupby(known["GRCODE"]).sum(),
})
assert len(lines) == 60 and (lines["n_ay"] == 10).sum() == 46  # 60 − 14 incomplete

p25, p75 = lines["size"].quantile([0.25, 0.75])
ok = (
    (lines["n_ay"] == 10)
    & lines["size"].between(p25, p75)
    & (lines["swing"] <= 0.5)
    & (lines["neg_case"] == 0)
)
eligible = lines[ok].sort_index()  # sorted, so the pick doesn't depend on row order
print(f"Size p25–p75: {p25:,.0f} to {p75:,.0f} ($000)")
print(f"Eligible: {len(eligible)} of {len(lines)}")
print(eligible.round(2))
assert len(eligible) >= 3, "Too few; decide in chat which criterion to loosen"

pick = np.random.default_rng(SEED).choice(eligible.index.to_numpy())
print(f"\nPick (seed {SEED}): GRCODE {pick}, {lines.loc[pick, 'GRNAME']}")

Size p25–p75: 1,464 to 29,021 ($000)
Eligible: 12 of 60
                                     GRNAME  n_ay     size  swing  neg_case
GRCODE                                                                     
671                   Farm Bureau Of MI Grp    10  20045.5   0.34         0
5940           American Contractors Ins Grp    10  28212.5   0.14         0
11126   Yasuda Fire & Marine Ins Co Of Amer    10  27411.5   0.34         0
13501                   Brethren Mut Ins Co    10   3792.0   0.44         0
14257                        IMT Ins Co Mut    10   5811.0   0.19         0
16446          Farmers Ins Co of Flemington    10   2174.5   0.18         0
18309              Pioneer State Mut Ins Co    10   1483.5   0.46         0
21172                       Vanliner Ins Co    10  17916.0   0.50         0
23663              National American Ins Co    10  16426.0   0.46         0
26433                     Harco Natl Ins Co    10   3652.5   0.34         0
40126                        Umi

# Step 3. Export the company

Splits GRCODE 23663's data so the Excel work only sees what was known at 12/31/2007?

- **Known workbook** (`excel/wc_23663.xlsx`): paid, reported and premium triangles for cells with AY + lag − 1 ≤ 2007 (55 per triangle). Steps 4–6 are built in this file.
- **Hidden answer key** (`data/hidden/wc_23663_hidden.csv`): the 45 cells after 2007. Not opened until step 10. Its SHA-256 hash is printed below, so at step 10 I can show the file wasn't changed.
- Values in $000. Paid = CumPaidLoss; reported = IncurredLosses − BulkLoss; premium = EarnedPremNet, one value per AY.

In [16]:
# Phase 2, step 3: export GRCODE 23663 known rows to Excel and hidden rows to a closed answer key
import hashlib
from pathlib import Path

GRCODE = 23663
VAL_YEAR = 2007  # valuation date 12/31/2007

co = wc_clean[wc_clean["GRCODE"] == GRCODE].copy()
assert co["GRNAME"].iloc[0].startswith("National American"), co["GRNAME"].iloc[0]
assert len(co) == 100  # complete line: 10 AYs x 10 lags

co["Reported"] = co["IncurredLosses"] - co["BulkLoss"]
co["CalYear"] = co["AccidentYear"] + co["DevelopmentLag"] - 1
is_known = co["CalYear"] <= VAL_YEAR

# Premium is one value per AY. Check it repeats on every row of the AY (reads premium only, no losses)
assert (co.groupby("AccidentYear")["EarnedPremNet"].nunique() == 1).all(), "Premium varies by lag; stop"

known, hidden = co[is_known], co[~is_known]
assert len(known) == 55 and len(hidden) == 45
assert known["CalYear"].max() == VAL_YEAR and hidden["CalYear"].min() == VAL_YEAR + 1

cols = ["AccidentYear", "DevelopmentLag", "CumPaidLoss", "IncurredLosses", "BulkLoss", "EarnedPremNet"]

info = pd.DataFrame([
    ("Company", f"{co['GRNAME'].iloc[0]} (GRCODE {GRCODE})"),
    ("Line", "Workers' compensation, net of reinsurance"),
    ("Source", "CAS Loss Reserving Database, AY 1998-2007 (cleaned: data/clean/wc_clean.csv)"),
    ("Valuation", "12/31/2007; known rows only (AY + lag - 1 <= 2007)"),
    ("Units", "$000"),
    ("Paid", "CumPaidLoss"),
    ("Reported", "IncurredLosses - BulkLoss (paid + case)"),
    ("Premium", "EarnedPremNet, one value per AY"),
], columns=["Item", "Value"])

xlsx = Path("../excel/wc_23663.xlsx")
xlsx.parent.mkdir(exist_ok=True)
assert not xlsx.exists(), f"{xlsx} exists; won't overwrite Excel work"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    info.to_excel(xw, sheet_name="Info", index=False)
    known[cols].sort_values(cols[:2]).to_excel(xw, sheet_name="Data", index=False)

key = Path("../data/hidden/wc_23663_hidden.csv")
key.parent.mkdir(parents=True, exist_ok=True)
hidden[["GRCODE", "AccidentYear", "DevelopmentLag", "CumPaidLoss", "Reported"]].to_csv(key, index=False)
sha = hashlib.sha256(key.read_bytes()).hexdigest()

del co, hidden  # nothing below this cell can show hidden values by accident
print(f"Known:  {len(known)} rows -> {xlsx}")
print(f"Hidden: 45 rows -> {key}")
print(f"Hidden SHA-256: {sha}")

AssertionError: ../excel/wc_23663.xlsx exists; won't overwrite Excel work

Export: 55 known rows -> excel/wc_23663.xlsx (Info, Data); 45 hidden rows -> data/hidden/wc_23663_hidden.csv, SHA-256 cc9c2b0a48ef282d…, not opened. Premium is the same at every lag for each AY, so lag 1 premium is the AY's premium.

# Industry ECR Calculation


In [ ]:
import pandas as pd
wc = pd.read_csv("../data/clean/wc_clean.csv")
a = wc[(wc.AccidentYear == 1998) & (wc.DevelopmentLag == 10)]
print("Number of clean company lines:",len(a))                                                                  # must be 60
print("Weighted-average paid ECR for all WC lines:",a.CumPaidLoss.sum() / a.EarnedPremNet.sum())                             # paid ELR → B4
print("Weighted-average reported ECR for all WC lines:",(a.IncurredLosses - a.BulkLoss).sum() / a.EarnedPremNet.sum())           # reported ELR, keep for step 5

Number of clean company lines: 60
Weighted-average paid ECR for all WC lines: 0.6418547575096493
Weighted-average reported ECR for all WC lines: 0.6816206316076523


# Step 8. Reconcile Python to Excel
Question: Does `src/methods.py` reproduce every number in `excel/wc_23663.xlsx`, stage by stage, to the dollar?

Why: Phase 3 runs this code on 169 company-lines that no one checks by hand. Matching totals isn't enough, because errors can offset. Comparing every stage (triangle → factors → averages → CDFs → ELRs → estimates → unpaid) shows the code makes the same calculation as the workbook, and a mismatch points to the stage that caused it.

What's compared, each basis (paid, reported):

| Stage | Excel source | Cells | Unit | Tolerance |
|---|---|---|---|---|
| Triangle | Triangle sheet, Part 1 (B7:K16) | 55 known, 45 blank | $000 | $1 |
| Earned premium | Earned Premium (B2:B11) | 10 | $000 | $1 |
| Age-to-age factors | Triangle sheet, Part 2 (B20:J29) | 45 | ratio | 1e-9 |
| 6 averages | Triangle sheet, Part 3 (B32:J37) | 54 | ratio | 1e-9 |
| CDFs to 120 | Triangle sheet, Part 4 (B42:K42) | 10 | ratio | 1e-9 |
| ELRs, company and industry | Projections sheet (B3:B4) | 2 | ratio | 1e-9 |
| Estimate at 120 by AY, 5 methods | Projections sheet, Parts A–C | 50 | $000 | $1 |
| Unpaid by AY, 5 methods | Projections sheet, Parts A–C | 50 | $000 | $1 |
| Total unpaid, 5 methods | Projections sheet, Part D totals (row 18) | 5 | $000 | $1 |

- **Tolerance:** amounts are in $000, so $1 = 0.001. Ratios should agree to floating-point precision; 1e-9 allows only for the reported industry ELR, which is typed in Excel (B4) to 9 decimals while Python computes it.
- **Blank cells:** the triangle and factor checks also require the same cells to be blank in both (hidden cells, AY + lag − 1 > 2007).
- **Not compared, inputs:** the selected factors (Part 4, row 40). They are judgment, not calculation, so Python reads them from the workbook and uses them as given.
- **Expected gap:** reported EC industry and BF industry differ by a few cents in total, from the 9-decimal industry ELR. Within tolerance, but not zero.
- **Hidden data:** Python reads `wc_clean.csv`, which holds every cell; `methods.py` filters to known cells before any calculation. The hidden answer key is not opened.

## 8.1 Setup
Loads the workbook's saved values and the clean data, and defines `check()` (compares one stage) and `reconcile()` (runs every stage for one basis). Prints the sheets found and how many of the company's rows Python uses: 55 known of 100, the rest hidden.

In [17]:
# Phase 2, step 8.1: setup - load both sources and define the check
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from src import methods as m

GRCODE = 23663
TOL_USD = 0.001    # $1, in $000
TOL_RATIO = 1e-9   # factors, averages, CDFs, ELRs
AYS = range(1998, 2008)

# data_only=True reads the values Excel last calculated (saved with the file), not the formulas
xl = load_workbook("../excel/wc_23663.xlsx", data_only=True)
wc = pd.read_csv("../data/clean/wc_clean.csv")
line = wc[wc.GRCODE == GRCODE]

# Excel layout per basis: sheet names and the Parts A-C columns for each method (rows 8-17 = AY 1998-2007)
LAYOUT = {
    "paid": {
        "tri": "Paid Triangle", "proj": "Paid Projections",
        "est":    {"cl": "E", "ec_company": "M", "ec_industry": "O", "bf_company": "Z",  "bf_industry": "AC"},
        "unpaid": {"cl": "F", "ec_company": "N", "ec_industry": "P", "bf_company": "Y",  "bf_industry": "AB"},
        "totals": {"cl": "AI", "ec_company": "AJ", "ec_industry": "AK", "bf_company": "AL", "bf_industry": "AM"},
    },
    "reported": {
        "tri": "Reported Triangle", "proj": "Reported Projections",
        "est":    {"cl": "E", "ec_company": "O", "ec_industry": "S", "bf_company": "AD", "bf_industry": "AG"},
        "unpaid": {"cl": "H", "ec_company": "R", "ec_industry": "U", "bf_company": "AE", "bf_industry": "AH"},
        "totals": {"cl": "AN", "ec_company": "AO", "ec_industry": "AP", "bf_company": "AQ", "bf_industry": "AR"},
    },
}
METHODS = ["cl", "ec_company", "ec_industry", "bf_company", "bf_industry"]


def xl_grid(sheet, rows, first_col, last_col):
    """Excel block -> float array; blank or "" cells become NaN."""
    ws = xl[sheet]
    vals = ws.iter_rows(min_row=rows[0], max_row=rows[-1], min_col=first_col,
                        max_col=last_col, values_only=True)
    return np.array([[np.nan if v in (None, "") else float(v) for v in r] for r in vals])


def xl_col(sheet, col, rows=range(8, 18)):
    """One column of the Projections sheet, AY 1998-2007, indexed by AY."""
    ws = xl[sheet]
    return pd.Series([float(ws[f"{col}{r}"].value) for r in rows], index=list(AYS))


def check(stage, py, ex, unit):
    """Compare Python to Excel cell by cell; one result row per stage."""
    py, ex = np.asarray(py, float).ravel(), np.asarray(ex, float).ravel()
    blank_mismatch = int((np.isnan(py) != np.isnan(ex)).sum())
    both = ~np.isnan(py) & ~np.isnan(ex)
    diff = np.abs(py[both] - ex[both]).max() if both.any() else 0.0
    tol = TOL_USD if unit == "$000" else TOL_RATIO
    return {"stage": stage, "unit": unit, "cells": int(both.sum()),
            "blank_mismatch": blank_mismatch,
            "max_abs_diff": f"${diff * 1000:,.2f}" if unit == "$000" else f"{diff:.1e}",
            "tolerance": "$1.00" if unit == "$000" else f"{TOL_RATIO:.0e}",
            "pass": bool(diff <= tol and blank_mismatch == 0)}


def reconcile(basis):
    """Run every stage for one basis. Returns (stage table, unpaid by AY: Python - Excel, in $)."""
    L = LAYOUT[basis]
    rows = []

    tri = m.triangle(line, basis)
    rows.append(check("Triangle", tri.to_numpy(), xl_grid(L["tri"], range(7, 17), 2, 11), "$000"))

    ep = m.premium(line)
    rows.append(check("Earned premium", ep.to_numpy(),
                      xl_grid("Earned Premium", range(2, 12), 2, 2), "$000"))

    rows.append(check("Age-to-age factors", m.ata_factors(tri).to_numpy(),
                      xl_grid(L["tri"], range(20, 30), 2, 10), "ratio"))

    py_avg = np.vstack([m.average(tri, kind, n).to_numpy() for kind, n in m.AVERAGES])
    rows.append(check("6 averages", py_avg, xl_grid(L["tri"], range(32, 38), 2, 10), "ratio"))

    selected = dict(enumerate(xl_grid(L["tri"], [40], 2, 10)[0], start=1))  # input, from Excel
    rows.append(check("CDFs to 120", m.cdfs(selected).to_numpy(),
                      xl_grid(L["tri"], [42], 2, 11)[0], "ratio"))

    elrs = {"company": m.elr_company(line, basis), "industry": m.elr_industry(wc, basis)}
    rows.append(check("ELRs", [elrs["company"], elrs["industry"]],
                      xl_grid(L["proj"], range(3, 5), 2, 2), "ratio"))

    proj = m.project(line, basis, selected, elrs)
    for kind in ["est", "unpaid"]:
        py_k = np.column_stack([proj[f"{kind}_{mt}"] for mt in METHODS])
        ex_k = np.column_stack([xl_col(L["proj"], L[kind][mt]) for mt in METHODS])
        label = "Estimate at 120 by AY" if kind == "est" else "Unpaid by AY"
        rows.append(check(label, py_k, ex_k, "$000"))

    py_tot = m.totals(proj)[[f"unpaid_{mt}" for mt in METHODS]]
    ex_tot = [xl[L["proj"]][f"{L['totals'][mt]}18"].value for mt in METHODS]
    rows.append(check("Total unpaid", py_tot, ex_tot, "$000"))

    by_ay = pd.DataFrame({mt: (proj[f"unpaid_{mt}"] - xl_col(L["proj"], L["unpaid"][mt])) * 1000
                          for mt in METHODS})
    return pd.DataFrame(rows), by_ay


print(f"Workbook sheets: {xl.sheetnames}")
print(f"GRCODE {GRCODE}: {len(m.known(line))} known rows used by Python (of {len(line)} in wc_clean.csv)")

Workbook sheets: ['Info', 'Data', 'Paid Triangle', 'Paid Projections', 'Reported Triangle', 'Reported Projections', 'Earned Premium', 'Loss Ratios']
GRCODE 23663: 55 known rows used by Python (of 100 in wc_clean.csv)


## 8.2 Paid basis
Output 1, **stage table**: one row per stage. `cells` = cells compared (both non-blank); `blank_mismatch` = cells blank in one source but not the other (must be 0); `max_abs_diff` = largest |Python − Excel| in the stage, in dollars for $000 stages; `pass` = max diff within tolerance and no blank mismatch.

Output 2, **unpaid by AY**: Python − Excel, in dollars, for each method. Shows where any gap sits, not just its size.

In [18]:
# Phase 2, step 8.2: paid basis
paid_stages, paid_by_ay = reconcile("paid")
print("PAID BASIS - stage checks (Python vs Excel)")
print(paid_stages.to_string(index=False))
print("\nPAID BASIS - unpaid by AY, Python - Excel ($)")
print(paid_by_ay.round(2).add(0.0).to_string())

PAID BASIS - stage checks (Python vs Excel)
                stage  unit  cells  blank_mismatch max_abs_diff tolerance  pass
             Triangle  $000     55               0        $0.00     $1.00  True
       Earned premium  $000     10               0        $0.00     $1.00  True
   Age-to-age factors ratio     45               0      0.0e+00     1e-09  True
           6 averages ratio     54               0      4.4e-16     1e-09  True
          CDFs to 120 ratio     10               0      4.4e-16     1e-09  True
                 ELRs ratio      2               0      2.2e-16     1e-09  True
Estimate at 120 by AY  $000     50               0        $0.00     $1.00  True
         Unpaid by AY  $000     50               0        $0.00     $1.00  True
         Total unpaid  $000      5               0        $0.00     $1.00  True

PAID BASIS - unpaid by AY, Python - Excel ($)
               cl  ec_company  ec_industry  bf_company  bf_industry
AccidentYear                             

## 8.3 Reported basis
Same two outputs as 8.2. Unpaid here = estimated reported at 120 − paid to date (case + IBNR), matching the Reported Projections sheet. The EC industry and BF industry columns are where the 9-decimal ELR gap shows up, if anywhere.

In [21]:
# Phase 2, step 8.3: reported basis
rep_stages, rep_by_ay = reconcile("reported")
print("REPORTED BASIS - stage checks (Python vs Excel)")
print(rep_stages.to_string(index=False))
print("\nREPORTED BASIS - unpaid by AY, Python - Excel ($)")
print(rep_by_ay.round(2).add(0.0).to_string())

REPORTED BASIS - stage checks (Python vs Excel)
                stage  unit  cells  blank_mismatch max_abs_diff tolerance  pass
             Triangle  $000     55               0        $0.00     $1.00  True
       Earned premium  $000     10               0        $0.00     $1.00  True
   Age-to-age factors ratio     45               0      0.0e+00     1e-09  True
           6 averages ratio     54               0      2.2e-16     1e-09  True
          CDFs to 120 ratio     10               0      2.2e-16     1e-09  True
                 ELRs ratio      2               0      3.9e-10     1e-09  True
Estimate at 120 by AY  $000     50               0        $0.01     $1.00  True
         Unpaid by AY  $000     50               0        $0.01     $1.00  True
         Total unpaid  $000      5               0        $0.07     $1.00  True

REPORTED BASIS - unpaid by AY, Python - Excel ($)
               cl  ec_company  ec_industry  bf_company  bf_industry
AccidentYear                     

## 8.4 Result
One line per basis, then a hard stop (`assert`) if any stage fails, so a broken reconciliation can't be skipped past when the notebook is rerun.

In [22]:
# Phase 2, step 8.4: overall result
for basis, stages in [("Paid", paid_stages), ("Reported", rep_stages)]:
    n_fail = (~stages["pass"]).sum()
    print(f"{basis:9s}: {len(stages) - n_fail} of {len(stages)} stages pass, "
          f"{stages.cells.sum()} cells compared")
failed = pd.concat([paid_stages.assign(basis="paid"), rep_stages.assign(basis="reported")])
failed = failed[~failed["pass"]]
assert failed.empty, f"Reconciliation failed:\n{failed.to_string(index=False)}"
print("\nPython reconciles to Excel within $1 at every stage, both bases.")

Paid     : 9 of 9 stages pass, 281 cells compared
Reported : 9 of 9 stages pass, 281 cells compared

Python reconciles to Excel within $1 at every stage, both bases.


Finding (draft; confirm against the output above): Python matches Excel at every stage on both bases (9 stages, 281 cells per basis). Paid basis: largest money difference $0.00; ratio differences ≤ 4.4e-16 (floating-point noise). Reported basis: unpaid differs by about $0.01 per AY and $0.07 in total, only in EC industry and BF industry, because the industry reported ELR is typed in Excel B4 to 9 decimals (diff 3.9e-10) while Python computes it. Within the $1 tolerance. `src/methods.py` reproduces the workbook and is ready for Phase 3.